# Tokenized Long-Horizon Spatio-Temporal CFD Forecasting

## Overview
This notebook implements a scalable pipeline for long-horizon forecasting of 2D URANS simulation fields
on a large unstructured mesh (~2.1M cells) using **spatial tokenization** and three temporal models:
- **LSTM** – per-token sequence model
- **Temporal Transformer** – attention over the time dimension only
- **Mamba-SSM** – selective state-space model (requires `mamba-ssm` package)

## Input Files
| Variable | Default | Description |
|---|---|---|
| `MESH_H5` | `mesh_coords.h5` | HDF5 file with cell-centred coordinates, shape `(N,2)` |
| `SIM_H5` | `simulation.h5` | HDF5 file with datasets `step_0000 … step_0999`, shape `(N,3)` – `[p, u, v]` |

## Quick-start
1. Set paths in **Configuration** cell below.
2. Choose model: `MODEL_CHOICE = 'lstm'` | `'transformer'` | `'mamba'`
3. Choose experiment: `EXPERIMENT = 'L100'` | `'L300'` | `'L500'`
4. Run all cells top-to-bottom (`Kernel → Restart & Run All`).

## Memory budget
The pipeline never loads the whole simulation into RAM.  
Token means are computed lazily and cached as a memory-mapped array.

In [ ]:
# ============================================================
# CELL 1 – CONFIGURATION  (edit here)
# ============================================================

# --- Input paths ---
MESH_H5 = 'mesh_coords.h5'          # HDF5 with coordinates (N,2)
SIM_H5  = 'simulation.h5'           # HDF5 with step_0000 … step_0999

# --- Tokenization ---
K           = 4096    # total number of spatial tokens
D_MODEL     = 128     # token embedding dimension

# Region allocation fractions (must sum to 1)
FRAC_NEAR   = 0.30
FRAC_GAPS   = 0.20
FRAC_WAKE   = 0.35
FRAC_FAR    = 0.15

# Solid-body mask
SOLID_EPS_PCTILE = 5      # percentile for low-speed threshold
SOLID_FRAC_THRESH = 0.98  # fraction of early steps below eps → solid
N_SOLID_CHECK_STEPS = 100 # how many early steps to scan for solid mask

# Near-body bandwidth (normalised coords)
D0_NEAR = 0.05

# Gap ROI y-margin and wake half-height (normalised coords)
GAP_Y_MARGIN   = 0.05
WAKE_X_MARGIN  = 0.05
WAKE_HALF_H    = 0.08

# --- Temporal setup ---
TOTAL_STEPS = 1000
DT          = 0.01   # seconds per step

# Time-block split indices
TRAIN_END = 700
VAL_END   = 850
# test: [850, 1000)

# --- Experiment (context length L, horizon H) ---
# 'L100' -> L=100, H=900 | 'L300' -> L=300, H=700 | 'L500' -> L=500, H=500
EXPERIMENT = 'L100'

_EXP_MAP = {'L100': (100, 900), 'L300': (300, 700), 'L500': (500, 500)}
CTX_LEN, HORIZON = _EXP_MAP[EXPERIMENT]

# --- Training ---
MODEL_CHOICE   = 'lstm'   # 'lstm' | 'transformer' | 'mamba'
ROLLOUT_STAGES = [20, 50, 100, 200]   # curriculum rollout lengths
TF_RATIOS      = [0.9, 0.7, 0.5, 0.3] # teacher-forcing ratio per stage
EPOCHS_PER_STAGE = 3
BATCH_SIZE     = 16
LR             = 1e-3
CLIP_GRAD      = 1.0
USE_AMP        = False  # mixed precision (requires CUDA)

# --- Tokenised data cache (written to /tmp to save disk quota) ---
CACHE_DIR = '/tmp/cfd_token_cache'

import os
os.makedirs(CACHE_DIR, exist_ok=True)
print('Configuration loaded.')
print(f'  Experiment : {EXPERIMENT}  CTX={CTX_LEN}  H={HORIZON}')
print(f'  Model      : {MODEL_CHOICE}')
print(f'  K tokens   : {K}')

In [ ]:
# ============================================================
# CELL 2 – IMPORTS & DEPENDENCY CHECKS
# ============================================================

import sys, os, time, math, warnings
import numpy as np
import h5py
from pathlib import Path
from scipy.spatial import cKDTree
from sklearn.cluster import MiniBatchKMeans

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Optional: mixed-precision
try:
    from torch.cuda.amp import autocast, GradScaler
    _AMP_AVAILABLE = True
except ImportError:
    _AMP_AVAILABLE = False

# Optional: mamba-ssm
MAMBA_AVAILABLE = False
try:
    from mamba_ssm import Mamba
    MAMBA_AVAILABLE = True
    print('mamba-ssm: AVAILABLE')
except ImportError:
    print('mamba-ssm: NOT FOUND')
    print('  To install:  pip install mamba-ssm causal-conv1d')
    print('  Falling back – LSTM or Transformer can still be run.')

if MODEL_CHOICE == 'mamba' and not MAMBA_AVAILABLE:
    print(f'WARNING: MODEL_CHOICE=mamba but mamba-ssm is not installed.')
    print('         Switching to lstm for this run.')
    MODEL_CHOICE = 'lstm'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# ============================================================
# CELL 3 – HDF5 UTILITIES
# ============================================================

def _walk_hdf5(h5file):
    """Return list of (path, shape) for every dataset in an HDF5 file."""
    results = []
    def _visitor(name, obj):
        if isinstance(obj, h5py.Dataset):
            results.append((name, obj.shape))
    h5file.visititems(_visitor)
    return results


def find_coords_key(mesh_path: str) -> str:
    """
    Locate the dataset key inside mesh_coords.h5 that has shape (N, 2).
    Returns the key string.
    """
    with h5py.File(mesh_path, 'r') as f:
        datasets = _walk_hdf5(f)
    print(f'mesh HDF5 datasets: {datasets}')
    candidates = [(k, s) for k, s in datasets if len(s) == 2 and s[1] == 2]
    if not candidates:
        raise KeyError(
            f'No (N,2) dataset found in {mesh_path}.\n'
            f'Available datasets: {datasets}')
    key, shape = candidates[0]
    print(f'Using coords key: "{key}"  shape={shape}')
    return key


def load_coords(mesh_path: str) -> np.ndarray:
    """Load cell-centred coordinates, shape (N, 2)."""
    key = find_coords_key(mesh_path)
    with h5py.File(mesh_path, 'r') as f:
        coords = f[key][()].astype(np.float32)
    return coords


def list_sim_steps(sim_path: str) -> list:
    """Return sorted list of dataset keys matching step_XXXX."""
    with h5py.File(sim_path, 'r') as f:
        keys = sorted([k for k in f.keys() if k.startswith('step_')])
    return keys


def load_step(sim_path: str, step_key: str) -> np.ndarray:
    """Load one simulation step, shape (N, 3)."""
    with h5py.File(sim_path, 'r') as f:
        data = f[step_key][()].astype(np.float32)
    return data


print('HDF5 utilities ready.')

In [ ]:
# ============================================================
# CELL 4 – SPATIAL TOKENISATION  (precompute once)
# ============================================================

import numpy as np
from scipy.spatial import cKDTree
from sklearn.cluster import MiniBatchKMeans


# ------------------------------------------------------------------
# 4a. Load & normalise coordinates
# ------------------------------------------------------------------
print('Loading coordinates …')
coords_raw = load_coords(MESH_H5)            # (N, 2)  physical coords
N_CELLS = coords_raw.shape[0]
print(f'  N_CELLS = {N_CELLS:,}')

coords_mean = coords_raw.mean(axis=0)
coords_std  = coords_raw.std(axis=0) + 1e-12
coords = (coords_raw - coords_mean) / coords_std   # normalised (N, 2)


# ------------------------------------------------------------------
# 4b. Solid-body mask
#     Cells where speed ≈ 0 for ≥98 % of the first 100 steps
# ------------------------------------------------------------------
print('Computing solid-body mask …')
sim_steps = list_sim_steps(SIM_H5)
assert len(sim_steps) >= N_SOLID_CHECK_STEPS, (
    f'Expected at least {N_SOLID_CHECK_STEPS} steps, got {len(sim_steps)}')

# Estimate eps from a subsample of cells × early steps
rng = np.random.default_rng(42)
sample_idx = rng.choice(N_CELLS, min(50_000, N_CELLS), replace=False)
speed_sample = []
for i in range(0, N_SOLID_CHECK_STEPS, 10):   # every 10th step → 10 reads
    d = load_step(SIM_H5, sim_steps[i])[sample_idx]  # (sub, 3)
    speed_sample.append(np.sqrt(d[:, 1]**2 + d[:, 2]**2))
speed_sample = np.concatenate(speed_sample)
eps = float(np.percentile(speed_sample, SOLID_EPS_PCTILE))
print(f'  Low-speed threshold eps = {eps:.4f}')

# Count low-speed events per cell
low_count = np.zeros(N_CELLS, dtype=np.int32)
for i in range(N_SOLID_CHECK_STEPS):
    d = load_step(SIM_H5, sim_steps[i])
    speed = np.sqrt(d[:, 1]**2 + d[:, 2]**2)
    low_count += (speed < eps).astype(np.int32)

solid_core = (low_count / N_SOLID_CHECK_STEPS) >= SOLID_FRAC_THRESH
print(f'  solid_core cells: {solid_core.sum():,}')


# ------------------------------------------------------------------
# 4c. Split solid_core into 3 elements via KMeans (slat/main/flap)
# ------------------------------------------------------------------
print('Clustering solid-core into 3 elements …')
solid_idx  = np.where(solid_core)[0]
solid_pts  = coords[solid_idx]          # (S, 2)  normalised

km3 = MiniBatchKMeans(n_clusters=3, random_state=42, n_init=5)
solid_labels = km3.fit_predict(solid_pts)
centroids = km3.cluster_centers_        # (3, 2)
order = np.argsort(centroids[:, 0])     # sort by x  → slat / main / flap
slat_id, main_id, flap_id = order
print(f'  Centroids (normalised): slat={centroids[slat_id]}, '
      f'main={centroids[main_id]}, flap={centroids[flap_id]}')

# Bounding boxes per element
def _bbox(idx_list):
    pts = coords[idx_list]
    return pts[:, 0].min(), pts[:, 0].max(), pts[:, 1].min(), pts[:, 1].max()

element_cells = []
for eid in order:
    mask_e = solid_labels == eid
    element_cells.append(solid_idx[mask_e])

bbox_slat = _bbox(element_cells[0])
bbox_main = _bbox(element_cells[1])
bbox_flap = _bbox(element_cells[2])


# ------------------------------------------------------------------
# 4d. ROI definitions (indices of fluid cells in each region)
# ------------------------------------------------------------------
print('Building ROIs …')

fluid_mask = ~solid_core
fluid_idx  = np.where(fluid_mask)[0]      # (F,)
fluid_pts  = coords[fluid_idx]            # (F, 2)

# --- near-body band ---
solid_tree = cKDTree(solid_pts)
dist_to_solid, _ = solid_tree.query(fluid_pts, workers=-1)
near_body_mask_local = dist_to_solid < D0_NEAR
near_body_idx = fluid_idx[near_body_mask_local]
print(f'  near-body cells: {len(near_body_idx):,}')

# --- gap ROIs (between element bounding boxes) ---
def _gap_roi(bbox_left, bbox_right, pts, fidx):
    x_lo = bbox_left[1]                          # right edge of left element
    x_hi = bbox_right[0]                         # left  edge of right element
    y_lo = min(bbox_left[2], bbox_right[2]) - GAP_Y_MARGIN
    y_hi = max(bbox_left[3], bbox_right[3]) + GAP_Y_MARGIN
    mask = ((pts[:, 0] > x_lo) & (pts[:, 0] < x_hi) &
            (pts[:, 1] > y_lo) & (pts[:, 1] < y_hi))
    return fidx[mask]

gap01_idx = _gap_roi(bbox_slat, bbox_main, fluid_pts, fluid_idx)
gap12_idx = _gap_roi(bbox_main, bbox_flap, fluid_pts, fluid_idx)
print(f'  gap 0-1 cells: {len(gap01_idx):,}   gap 1-2: {len(gap12_idx):,}')

# --- wake ROIs (per-element + global corridor) ---
def _wake_roi(bbox, pts, fidx, x_margin=WAKE_X_MARGIN, half_h=WAKE_HALF_H):
    x_te  = bbox[1]                              # trailing edge x
    y_mid = (bbox[2] + bbox[3]) / 2
    mask  = ((pts[:, 0] > x_te - x_margin) &
             (np.abs(pts[:, 1] - y_mid) < half_h))
    return fidx[mask]

wake_slat_idx = _wake_roi(bbox_slat, fluid_pts, fluid_idx)
wake_main_idx = _wake_roi(bbox_main, fluid_pts, fluid_idx)
wake_flap_idx = _wake_roi(bbox_flap, fluid_pts, fluid_idx)
print(f'  wake cells: slat={len(wake_slat_idx):,}  '
      f'main={len(wake_main_idx):,}  flap={len(wake_flap_idx):,}')


# ------------------------------------------------------------------
# 4e. Allocate token budgets & cluster per region
# ------------------------------------------------------------------
print('Allocating token budgets …')

# Build region index sets (fluid cells only; deduplicate)
def _dedup(a, *exclude_sets):
    """Return a without elements in any of exclude_sets."""
    excl = np.concatenate(exclude_sets) if exclude_sets else np.array([], dtype=np.int64)
    excl_set = set(excl.tolist())
    return a[~np.isin(a, list(excl_set))]

gap_idx  = np.unique(np.concatenate([gap01_idx, gap12_idx]))
wake_idx = np.unique(np.concatenate([wake_slat_idx, wake_main_idx, wake_flap_idx]))
# near_body_idx already computed; remove overlaps going broad→narrow
wake_idx = _dedup(wake_idx, near_body_idx)
gap_idx  = _dedup(gap_idx, near_body_idx, wake_idx)
# far-field = all remaining fluid
assigned = np.unique(np.concatenate([near_body_idx, gap_idx, wake_idx]))
far_idx  = _dedup(fluid_idx, assigned)

k_near = max(1, round(K * FRAC_NEAR))
k_gaps = max(1, round(K * FRAC_GAPS))
k_wake = max(1, round(K * FRAC_WAKE))
k_far  = K - k_near - k_gaps - k_wake
print(f'  token budget: near={k_near}  gaps={k_gaps}  wake={k_wake}  far={k_far}')


def _cluster_region(idx, k, seed=42):
    """MiniBatchKMeans on normalised coords of idx; return (labels, centers)."""
    pts = coords[idx]
    km = MiniBatchKMeans(n_clusters=k, random_state=seed, n_init=5,
                         batch_size=min(10_000, len(idx)))
    labels = km.fit_predict(pts)
    return labels, km.cluster_centers_


print('Clustering near-body …')
labels_near,  centers_near  = _cluster_region(near_body_idx, k_near)
print('Clustering gaps …')
labels_gaps,  centers_gaps  = _cluster_region(gap_idx,       k_gaps)
print('Clustering wakes …')
labels_wake,  centers_wake  = _cluster_region(wake_idx,      k_wake)
print('Clustering far-field …')
labels_far,   centers_far   = _cluster_region(far_idx,       k_far)


# ------------------------------------------------------------------
# 4f. Build global cluster_id array (N_CELLS,) and membership CSR
# ------------------------------------------------------------------
print('Building global cluster_id …')

# Offset each region's labels
offset_gaps = k_near
offset_wake = k_near + k_gaps
offset_far  = k_near + k_gaps + k_wake

cluster_id = np.full(N_CELLS, -1, dtype=np.int32)  # -1 = solid
cluster_id[near_body_idx] = labels_near
cluster_id[gap_idx]       = labels_gaps + offset_gaps
cluster_id[wake_idx]      = labels_wake + offset_wake
cluster_id[far_idx]       = labels_far  + offset_far

# Global cluster centres (K, 2)
cluster_centers = np.vstack([
    centers_near, centers_gaps, centers_wake, centers_far
]).astype(np.float32)   # (K, 2)

assert cluster_id.max() == K - 1, f'Expected max id {K-1}, got {cluster_id.max()}'
print(f'  cluster_id range: [{cluster_id.min()}, {cluster_id.max()}]')


# ------------------------------------------------------------------
# 4g. CSR-style membership for efficient scatter-mean
# ------------------------------------------------------------------
print('Building CSR membership map …')

# Sort fluid cells by cluster id → CSR indptr
fluid_cid      = cluster_id[fluid_idx]    # (F,)
sort_order     = np.argsort(fluid_cid, kind='stable')
sorted_fluid   = fluid_idx[sort_order]     # cell global indices in cluster order
sorted_cid     = fluid_cid[sort_order]

indptr = np.zeros(K + 1, dtype=np.int64)
for cid in sorted_cid:
    indptr[cid + 1] += 1
np.cumsum(indptr, out=indptr)
# sorted_fluid[indptr[k]:indptr[k+1]] = cell indices belonging to token k

print(f'  CSR ready. Total fluid cells assigned: {len(sorted_fluid):,}')
print('Spatial tokenisation complete.')

In [ ]:
# ============================================================
# CELL 5 – TOKENISATION FUNCTION  (step → token means)
#          + FULL-SIM CACHE
# ============================================================

def step_to_token_means(field: np.ndarray) -> np.ndarray:
    """
    Parameters
    ----------
    field : (N_CELLS, 3)  float32  [p, u, v]

    Returns
    -------
    token_means : (K, 3)  float32
    """
    token_means = np.zeros((K, 3), dtype=np.float32)
    counts      = indptr[1:] - indptr[:-1]          # (K,)  cells per token
    # Vectorised scatter-sum via numpy.add.at
    np.add.at(token_means, sorted_cid, field[sorted_fluid])
    # Avoid division by zero for empty tokens
    safe_counts = np.maximum(counts, 1)[:, None]
    token_means /= safe_counts
    return token_means


# --- Build or load the token cache ---
TOKEN_CACHE = os.path.join(CACHE_DIR, f'token_means_K{K}.npy')

if os.path.exists(TOKEN_CACHE):
    print(f'Loading cached token means from {TOKEN_CACHE} …')
    token_series = np.load(TOKEN_CACHE, mmap_mode='r')   # (T, K, 3)
    T_CACHED = token_series.shape[0]
    print(f'  shape: {token_series.shape}')
else:
    print(f'Computing token means for all {TOTAL_STEPS} steps …')
    token_series = np.zeros((TOTAL_STEPS, K, 3), dtype=np.float32)
    t0 = time.time()
    for i, key in enumerate(sim_steps[:TOTAL_STEPS]):
        field = load_step(SIM_H5, key)                    # (N, 3)
        token_series[i] = step_to_token_means(field)
        if (i + 1) % 100 == 0:
            print(f'  {i+1}/{TOTAL_STEPS}  ({time.time()-t0:.1f}s)')
    np.save(TOKEN_CACHE, token_series)
    print(f'Saved to {TOKEN_CACHE}')

T_CACHED = token_series.shape[0]
print(f'token_series shape: {token_series.shape}  dtype: {token_series.dtype}')

In [ ]:
# ============================================================
# CELL 6 – NORMALISATION STATISTICS
# ============================================================

# Compute mean and std from training portion only
train_tokens = token_series[:TRAIN_END]          # (700, K, 3)
tok_mean = train_tokens.mean(axis=(0, 1), keepdims=True)   # (1, 1, 3)
tok_std  = train_tokens.std(axis=(0, 1), keepdims=True) + 1e-8
print(f'Token mean: {tok_mean.squeeze()}')
print(f'Token std : {tok_std.squeeze()}')

def normalise(x):   return (x - tok_mean) / tok_std
def denormalise(x): return x * tok_std + tok_mean

token_series_norm = normalise(token_series).astype(np.float32)  # (T, K, 3)
print('Normalisation done.')

In [ ]:
# ============================================================
# CELL 7 – DATASET & DATALOADER
#          Sliding-window, time-block split, no leakage
# ============================================================

class TokenWindowDataset(Dataset):
    """
    Sliding window dataset over a time segment of token_series.

    Returns
    -------
    x : (window, K, 3)   input sequence
    y : (window, K, 3)   next-step targets  (x shifted by 1)
    """
    def __init__(self, series: np.ndarray, window: int,
                 start: int = 0, end: int = None):
        end = end if end is not None else len(series)
        self.data   = series[start:end]   # view
        self.window = window
        self.n = len(self.data) - window  # number of valid windows
        assert self.n > 0, f'Segment too short: {len(self.data)} < window {window}'

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        x = torch.from_numpy(self.data[idx     : idx + self.window].copy())
        y = torch.from_numpy(self.data[idx + 1 : idx + self.window + 1].copy())
        return x, y


def make_dataloaders(window: int, batch_size: int = BATCH_SIZE,
                     num_workers: int = 0):
    """Return train_loader, val_loader for the given rollout window."""
    train_ds = TokenWindowDataset(token_series_norm, window, 0, TRAIN_END)
    val_ds   = TokenWindowDataset(token_series_norm, window, TRAIN_END, VAL_END)
    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True,  drop_last=True,
                              num_workers=num_workers)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size,
                              shuffle=False, drop_last=False,
                              num_workers=num_workers)
    print(f'  window={window}: train={len(train_ds)} val={len(val_ds)} windows')
    return train_loader, val_loader


print('Dataset classes ready.')

In [ ]:
# ============================================================
# CELL 8 – TOKEN COORDINATE EMBEDDING
# ============================================================

class CoordEmbedding(nn.Module):
    """Small MLP: (K, 2) → (K, d_model).  Applied once, frozen after build."""
    def __init__(self, d_model: int = D_MODEL):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, coords_t: torch.Tensor) -> torch.Tensor:
        """coords_t: (K, 2) → (K, d_model)."""
        return self.net(coords_t)


# Pre-compute coord embedding as a fixed tensor
_coord_emb_module = CoordEmbedding(D_MODEL)
with torch.no_grad():
    _cc_t = torch.from_numpy(cluster_centers)         # (K, 2)
    COORD_EMBD = _coord_emb_module(_cc_t)             # (K, D_MODEL)
COORD_EMBD = COORD_EMBD.to(DEVICE)                    # fixed (K, D_MODEL)

print(f'COORD_EMBD shape: {COORD_EMBD.shape}')

In [ ]:
# ============================================================
# CELL 9 – MODEL DEFINITIONS
#          LSTM | Temporal Transformer | Mamba-SSM
#          All share the same input projection and output head.
# ============================================================

class InputProj(nn.Module):
    """Project token field [p,u,v] + coord embedding → d_model."""
    def __init__(self, d_model: int = D_MODEL):
        super().__init__()
        self.field_proj = nn.Linear(3, d_model)
        self.fuse       = nn.Linear(d_model * 2, d_model)

    def forward(self, x: torch.Tensor, coord_embd: torch.Tensor) -> torch.Tensor:
        """
        x          : (B, T, K, 3)
        coord_embd : (K, d_model)
        → out      : (B*K, T, d_model)
        """
        B, T, K, _ = x.shape
        fp  = self.field_proj(x)                        # (B, T, K, d)
        ce  = coord_embd.unsqueeze(0).unsqueeze(0)      # (1, 1, K, d)
        ce  = ce.expand(B, T, -1, -1)
        fused = torch.cat([fp, ce], dim=-1)             # (B, T, K, 2d)
        out = self.fuse(fused)                          # (B, T, K, d)
        # Merge batch & token dims for temporal models
        out = out.permute(0, 2, 1, 3).reshape(B * K, T, D_MODEL)
        return out


class OutputHead(nn.Module):
    """Project d_model → 3 (p, u, v) per token per step."""
    def __init__(self, d_model: int = D_MODEL):
        super().__init__()
        self.net = nn.Linear(d_model, 3)

    def forward(self, h: torch.Tensor, B: int, K: int) -> torch.Tensor:
        """
        h   : (B*K, T, d_model)
        → out : (B, T, K, 3)
        """
        T = h.shape[1]
        out = self.net(h)                               # (B*K, T, 3)
        return out.reshape(B, K, T, 3).permute(0, 2, 1, 3)


# --- LSTM core ---
class LSTMCore(nn.Module):
    def __init__(self, d_model: int = D_MODEL, n_layers: int = 2,
                 dropout: float = 0.1):
        super().__init__()
        self.lstm = nn.LSTM(d_model, d_model, num_layers=n_layers,
                            batch_first=True, dropout=dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B*K, T, d)  → out: (B*K, T, d)."""
        out, _ = self.lstm(x)
        return out


# --- Temporal Transformer core ---
class TemporalTransformerCore(nn.Module):
    def __init__(self, d_model: int = D_MODEL, n_heads: int = 4,
                 n_layers: int = 2, dim_ff: int = 256, dropout: float = 0.1,
                 max_len: int = 1024):
        super().__init__()
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        # Learnable positional encoding
        self.pos_emb = nn.Embedding(max_len, d_model)

    def _causal_mask(self, T: int, device) -> torch.Tensor:
        mask = torch.triu(torch.ones(T, T, device=device), diagonal=1).bool()
        return mask

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B*K, T, d)  → out: (B*K, T, d)."""
        T = x.shape[1]
        pos = torch.arange(T, device=x.device)
        x = x + self.pos_emb(pos)
        mask = self._causal_mask(T, x.device)
        return self.encoder(x, mask=mask, is_causal=True)


# --- Mamba-SSM core ---
class MambaCore(nn.Module):
    def __init__(self, d_model: int = D_MODEL, n_layers: int = 2,
                 d_state: int = 16, d_conv: int = 4, expand: int = 2):
        super().__init__()
        assert MAMBA_AVAILABLE, 'mamba-ssm not installed'
        layers = []
        for _ in range(n_layers):
            layers.append(Mamba(d_model=d_model, d_state=d_state,
                                d_conv=d_conv, expand=expand))
        self.layers = nn.ModuleList(layers)
        self.norm   = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B*K, T, d)  → out: (B*K, T, d)."""
        for layer in self.layers:
            x = x + layer(self.norm(x))
        return x


# --- Unified wrapper ---
class TemporalForecaster(nn.Module):
    """
    Shared architecture:
      InputProj  →  temporal core  →  OutputHead

    All models predict next-step token values auto-regressively.
    """
    def __init__(self, model_type: str = MODEL_CHOICE, d_model: int = D_MODEL,
                 K: int = K):
        super().__init__()
        self.K          = K
        self.model_type = model_type
        self.input_proj = InputProj(d_model)
        self.output_head = OutputHead(d_model)

        if model_type == 'lstm':
            self.core = LSTMCore(d_model)
        elif model_type == 'transformer':
            self.core = TemporalTransformerCore(d_model)
        elif model_type == 'mamba':
            self.core = MambaCore(d_model)
        else:
            raise ValueError(f'Unknown model_type: {model_type}')

    def forward(self, x: torch.Tensor, coord_embd: torch.Tensor) -> torch.Tensor:
        """
        x          : (B, T, K, 3)
        coord_embd : (K, d_model)
        → preds    : (B, T, K, 3)  predictions for each next step
        """
        B = x.shape[0]
        h = self.input_proj(x, coord_embd)    # (B*K, T, d)
        h = self.core(h)                       # (B*K, T, d)
        return self.output_head(h, B, self.K)  # (B, T, K, 3)


# Quick shape check
_model_check = TemporalForecaster(MODEL_CHOICE, D_MODEL, K).to(DEVICE)
_x_check = torch.zeros(2, 5, K, 3, device=DEVICE)
_out_check = _model_check(_x_check, COORD_EMBD)
print(f'Model output shape: {_out_check.shape}  (expected (2,5,{K},3))')
del _model_check, _x_check, _out_check

In [ ]:
# ============================================================
# CELL 10 – TRAINING LOOP
#           Rollout curriculum + scheduled sampling
# ============================================================

def rollout_step(model: nn.Module,
                 ctx: torch.Tensor,
                 target: torch.Tensor,
                 rollout_len: int,
                 tf_ratio: float,
                 coord_embd: torch.Tensor) -> torch.Tensor:
    """
    Auto-regressive rollout with scheduled sampling.

    Parameters
    ----------
    ctx        : (B, CTX, K, 3)   ground-truth context window
    target     : (B, CTX+rollout_len, K, 3)  full ground-truth (for teacher forcing)
    rollout_len: number of free-running steps
    tf_ratio   : probability of using ground-truth at each step

    Returns
    -------
    preds : (B, rollout_len, K, 3)
    """
    B = ctx.shape[0]
    CTX = ctx.shape[1]
    preds = []
    window = ctx.clone()                    # (B, CTX, K, 3)  rolling input

    for t in range(rollout_len):
        out = model(window, coord_embd)     # (B, CTX, K, 3)
        next_pred = out[:, -1:, :, :]       # (B, 1, K, 3)  last step prediction
        preds.append(next_pred)

        # Scheduled sampling: choose next input
        if torch.rand(1).item() < tf_ratio:
            # Teacher forcing: use ground truth
            next_input = target[:, CTX + t : CTX + t + 1, :, :]
        else:
            next_input = next_pred.detach()

        window = torch.cat([window[:, 1:, :, :], next_input], dim=1)

    return torch.cat(preds, dim=1)          # (B, rollout_len, K, 3)


def train_epoch(model, loader, optimiser, scaler, rollout_len, tf_ratio):
    model.train()
    total_loss = 0.0
    for batch_x, batch_y in loader:
        # batch_x: (B, window, K, 3), batch_y: (B, window, K, 3)
        B, W, Kk, _ = batch_x.shape
        # Use first CTX_LEN steps as context (clip window if shorter)
        ctx_len = min(CTX_LEN, W - rollout_len)
        if ctx_len <= 0:
            continue
        ctx    = batch_x[:, :ctx_len].to(DEVICE)
        target = batch_x.to(DEVICE)          # for teacher forcing lookups
        gt     = batch_y[:, ctx_len - 1 : ctx_len - 1 + rollout_len].to(DEVICE)

        optimiser.zero_grad()
        if USE_AMP and _AMP_AVAILABLE:
            with autocast():
                preds = rollout_step(model, ctx, target, rollout_len, tf_ratio,
                                     COORD_EMBD)
                loss = F.mse_loss(preds, gt)
            scaler.scale(loss).backward()
            scaler.unscale_(optimiser)
            nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
            scaler.step(optimiser)
            scaler.update()
        else:
            preds = rollout_step(model, ctx, target, rollout_len, tf_ratio,
                                 COORD_EMBD)
            loss = F.mse_loss(preds, gt)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
            optimiser.step()

        total_loss += loss.item()
    return total_loss / max(len(loader), 1)


@torch.no_grad()
def val_epoch(model, loader, rollout_len):
    model.eval()
    total_loss = 0.0
    for batch_x, batch_y in loader:
        B, W, Kk, _ = batch_x.shape
        ctx_len = min(CTX_LEN, W - rollout_len)
        if ctx_len <= 0:
            continue
        ctx    = batch_x[:, :ctx_len].to(DEVICE)
        target = batch_x.to(DEVICE)
        gt     = batch_y[:, ctx_len - 1 : ctx_len - 1 + rollout_len].to(DEVICE)
        preds  = rollout_step(model, ctx, target, rollout_len, tf_ratio=1.0,
                              coord_embd=COORD_EMBD)  # full TF for val
        loss   = F.mse_loss(preds, gt)
        total_loss += loss.item()
    return total_loss / max(len(loader), 1)


# ---- Training orchestration ----
print(f'Building model: {MODEL_CHOICE}')
model     = TemporalForecaster(MODEL_CHOICE, D_MODEL, K).to(DEVICE)
optimiser = torch.optim.AdamW(model.parameters(), lr=LR)
scaler    = GradScaler() if (USE_AMP and _AMP_AVAILABLE) else None

n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')

train_history = []   # (stage, epoch, train_loss, val_loss)

for stage_idx, (rollout_len, tf_ratio) in enumerate(
        zip(ROLLOUT_STAGES, TF_RATIOS)):

    window = CTX_LEN + rollout_len
    if window > TRAIN_END:
        print(f'Stage {stage_idx}: window={window} > TRAIN_END, skipping.')
        continue

    train_loader, val_loader = make_dataloaders(window)
    print(f'\n=== Stage {stage_idx}: rollout={rollout_len}  tf={tf_ratio} ===')

    for ep in range(EPOCHS_PER_STAGE):
        t0 = time.time()
        tr_loss = train_epoch(model, train_loader, optimiser, scaler,
                              rollout_len, tf_ratio)
        vl_loss = val_epoch(model, val_loader, rollout_len)
        elapsed = time.time() - t0
        print(f'  ep {ep+1}/{EPOCHS_PER_STAGE}  '
              f'train={tr_loss:.4f}  val={vl_loss:.4f}  ({elapsed:.1f}s)')
        train_history.append((stage_idx, ep, tr_loss, vl_loss))

print('\nTraining complete.')

In [ ]:
# ============================================================
# CELL 11 – FULL-HORIZON EVALUATION
#           Rollout on test window, RMSE curve, vortex frequency
# ============================================================

# ------------------------------------------------------------------
# 11a. Full-horizon rollout on test segment
# ------------------------------------------------------------------
TEST_START = VAL_END   # 850
TEST_END   = TOTAL_STEPS  # 1000

# Context window: last CTX_LEN steps before test
ctx_start = TEST_START - CTX_LEN
assert ctx_start >= 0, f'CTX_LEN={CTX_LEN} exceeds test start {TEST_START}'

ctx_np = token_series_norm[ctx_start : TEST_START]   # (CTX_LEN, K, 3)
ctx_t  = torch.from_numpy(ctx_np).unsqueeze(0).to(DEVICE)  # (1, CTX_LEN, K, 3)

model.eval()
pred_token_list = []
window_t = ctx_t.clone()

with torch.no_grad():
    for step_i in range(TEST_END - TEST_START):
        out     = model(window_t, COORD_EMBD)       # (1, CTX_LEN, K, 3)
        nxt     = out[:, -1:, :, :]                 # (1, 1, K, 3)
        pred_token_list.append(nxt.cpu().numpy())   # append (1,1,K,3)
        window_t = torch.cat([window_t[:, 1:, :, :], nxt], dim=1)

# Stack predictions: (H, K, 3)  in normalised space
pred_token_norm = np.concatenate(pred_token_list, axis=1)[0]   # (H, K, 3)
pred_token = denormalise(pred_token_norm)             # (H, K, 3)  physical
true_token = denormalise(token_series_norm[TEST_START:TEST_END])  # (H, K, 3)

H_EVAL = pred_token.shape[0]
print(f'Rollout done.  H={H_EVAL} steps  shape: {pred_token.shape}')


# ------------------------------------------------------------------
# 11b. Full-field RMSE curve
#      Scatter token predictions back to all fluid cells.
# ------------------------------------------------------------------
print('Computing full-field RMSE …')

# Identify non-solid fluid cluster ids
fluid_cid_all = cluster_id[fluid_idx]   # (F,)  cluster ids of fluid cells

rmse_curve = np.zeros(H_EVAL, dtype=np.float32)
rmse_by_var = np.zeros((H_EVAL, 3), dtype=np.float32)  # p, u, v

for h in range(H_EVAL):
    # Scatter token predictions to fluid cells (piecewise constant)
    pred_cells = pred_token[h][fluid_cid_all]   # (F, 3)

    # Load ground truth for this test step
    step_key    = sim_steps[TEST_START + h]
    true_cells  = load_step(SIM_H5, step_key)[fluid_idx]   # (F, 3)

    diff = pred_cells - true_cells                          # (F, 3)
    rmse_by_var[h] = np.sqrt((diff**2).mean(axis=0))
    rmse_curve[h]  = float(np.sqrt((diff**2).mean()))

print(f'  Mean RMSE over {H_EVAL} steps:')
print(f'    p={rmse_by_var[:,0].mean():.4f}  '
      f'u={rmse_by_var[:,1].mean():.4f}  '
      f'v={rmse_by_var[:,2].mean():.4f}')
print(f'    Overall mean RMSE = {rmse_curve.mean():.4f}')


# ------------------------------------------------------------------
# 11c. Vortex-frequency accuracy
# ------------------------------------------------------------------
print('\nComputing vortex-frequency metrics …')

# Probe token sets: wake tokens downstream of each element TE
def _probe_tokens_for_wake(wake_cell_idx):
    """Return unique cluster ids present in a wake region."""
    cids = cluster_id[wake_cell_idx]
    return np.unique(cids[cids >= 0])

probe_slat  = _probe_tokens_for_wake(wake_slat_idx)
probe_main  = _probe_tokens_for_wake(wake_main_idx)
probe_flap  = _probe_tokens_for_wake(wake_flap_idx)
probe_gap01 = _probe_tokens_for_wake(gap01_idx)
probe_gap12 = _probe_tokens_for_wake(gap12_idx)

probe_sets = {
    'wake_slat':  probe_slat,
    'wake_main':  probe_main,
    'wake_flap':  probe_flap,
    'gap_01':     probe_gap01,
    'gap_12':     probe_gap12,
}

# Build full series (train + test) for each probe set
# Use token_series (physical, unnormalised already loaded)
T_TOTAL = token_series.shape[0]

freq_results = {}

for name, probe_ids in probe_sets.items():
    if len(probe_ids) == 0:
        print(f'  {name}: no probe tokens, skipping.')
        continue

    # Mean u-velocity over probe tokens, full time series
    u_series_all  = token_series[:, probe_ids, 1].mean(axis=1)   # (T,)
    # Predicted test segment
    u_pred_test   = pred_token[:, probe_ids, 1].mean(axis=1)      # (H,)
    u_true_test   = token_series[TEST_START:TEST_END, probe_ids, 1].mean(axis=1)

    def _dominant_freq(signal, dt=DT):
        """Return dominant frequency (Hz) of a 1-D signal."""
        n   = len(signal)
        sig = signal - signal.mean()
        fft = np.abs(np.fft.rfft(sig))
        freq = np.fft.rfftfreq(n, d=dt)
        # Ignore DC
        fft[0] = 0
        return float(freq[np.argmax(fft)])

    f_true = _dominant_freq(u_true_test)
    f_pred = _dominant_freq(u_pred_test)
    abs_err = abs(f_pred - f_true)
    rel_err = abs_err / max(f_true, 1e-12) * 100
    freq_results[name] = dict(f_true=f_true, f_pred=f_pred,
                               abs_err=abs_err, rel_err_pct=rel_err)
    print(f'  {name:12s}: f_true={f_true:.3f} Hz  f_pred={f_pred:.3f} Hz  '
          f'|err|={abs_err:.3f} Hz  rel={rel_err:.1f}%')

print('\nEvaluation complete.')

In [ ]:
# ============================================================
# CELL 12 – RESULTS SUMMARY & PLOTS
# ============================================================

try:
    import matplotlib
    matplotlib.use('Agg')   # non-interactive backend (safe on headless machines)
    import matplotlib.pyplot as plt
    _MPL = True
except ImportError:
    _MPL = False
    print('matplotlib not available – skipping plots.')

t_axis = np.arange(H_EVAL) * DT   # seconds

print(f'\n====  EXPERIMENT: {EXPERIMENT}  |  MODEL: {MODEL_CHOICE}  ====')
print(f'Context length : {CTX_LEN} steps ({CTX_LEN*DT:.2f}s)')
print(f'Horizon        : {H_EVAL}  steps ({H_EVAL*DT:.2f}s)')
print(f'Mean RMSE      : {rmse_curve.mean():.6f}')
print(f'Max  RMSE      : {rmse_curve.max():.6f}  (at t={t_axis[rmse_curve.argmax()]:.2f}s)')

print('\nVortex-frequency results:')
for name, r in freq_results.items():
    print(f'  {name:12s}: f_true={r["f_true"]:.3f} Hz  '
          f'f_pred={r["f_pred"]:.3f} Hz  rel_err={r["rel_err_pct"]:.1f}%')

if _MPL:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # RMSE curve
    axes[0].plot(t_axis, rmse_curve, color='steelblue', lw=1)
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('RMSE')
    axes[0].set_title(f'Full-field RMSE  [{EXPERIMENT} / {MODEL_CHOICE}]')
    axes[0].grid(True, alpha=0.4)

    # Per-variable RMSE
    for j, vname in enumerate(['p', 'u', 'v']):
        axes[1].plot(t_axis, rmse_by_var[:, j], label=vname, lw=1)
    axes[1].set_xlabel('Time (s)')
    axes[1].set_ylabel('RMSE')
    axes[1].set_title('Per-variable RMSE')
    axes[1].legend()
    axes[1].grid(True, alpha=0.4)

    plt.tight_layout()
    plot_path = os.path.join(CACHE_DIR,
                             f'rmse_{EXPERIMENT}_{MODEL_CHOICE}.png')
    plt.savefig(plot_path, dpi=120)
    plt.show()
    print(f'Plot saved to {plot_path}')

# Expose key results as variables for downstream use
RESULTS = dict(
    experiment  = EXPERIMENT,
    model       = MODEL_CHOICE,
    ctx_len     = CTX_LEN,
    horizon     = H_EVAL,
    rmse_curve  = rmse_curve.tolist(),
    mean_rmse   = float(rmse_curve.mean()),
    freq_results = freq_results,
)
print('\nRESULTS dict populated (available as RESULTS variable).')